Procedural Terrain Morphing ,  β-VAE (Colab)
Train a Variational Autoencoder on real elevation patches, then export **only the decoder**
to ONNX for the Godot slider UI.

Run order:
  1. Run `tile_scraper.py` locally (see README) -> produces `patches.npy` (+ preview.png).
     Eyeball preview.png first: you want mountains/ridges, not ocean.
  2. Upload `patches.npy` to Colab (`/content/`) when the notebook asks.
  3. Runtime -> Change runtime type -> GPU, then Run all.

In [ ]:
!pip install -q lightning onnx onnxruntime imageio  # uncomment if imports fail

import os, math, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
!python /kaggle/input/datasets/sohampansare/tile-scraper/tile_scraper.py --regions himalayas andes --zoom 12 --target-patches 2000 --max-tiles 300 --out /kaggle/working/data/


In [ ]:
CFG = dict(
    patches_path = "kaggle/working/data/patches.npy",  # upload the scraper output here
    img_size     = 64,
    latent_dim   = 5,     # 5 sliders in Godot. If morphs look entangled, try 16 first to confirm the pipeline, then come back to 5.
    beta         = 4.0,   # β-VAE weight: higher = more disentangled sliders, but blurrier reconstructions
    kl_warmup_epochs = 15,
    lr           = 1e-3,
    batch_size   = 64,
    max_epochs   = 60,
    seed         = 42,
    val_frac     = 0.1,
)

def seed_all(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["seed"])

In [ ]:
import os
print("cwd:", os.getcwd())
print("npy files:", [f for f in os.listdir('/kaggle/working/data') if f.endswith('.npy')])

In [ ]:
if not os.path.exists(CFG["patches_path"]):
    from google.colab import files
    print("Upload patches.npy from the tile scraper:")
    files.upload()  # then move it to /content/ if needed

raw = np.load(CFG["patches_path"]).astype(np.float32)  # (N, H, W) in METRES
print("loaded:", raw.shape, f"elev {raw.min():.0f}..{raw.max():.0f} m")

# Per-patch min-max -> [0, 1]. Each patch uses its full dynamic range,
# so every latent position decodes to terrain with real relief (no flat outputs).
mn = raw.reshape(len(raw), -1).min(1)[:, None, None]
mx = raw.reshape(len(raw), -1).max(1)[:, None, None]
rng = np.clip(mx - mn, 1e-6, None)
X = (raw - mn) / rng
X = X[(mx - mn).ravel() > 1.0]  # drop degenerate patches
print("kept patches:", len(X))

X = torch.from_numpy(X[:, None])  # (N, 1, H, W)
ds = TensorDataset(X, X)
n_val = max(1, int(len(ds) * CFG["val_frac"]))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val],
                                generator=torch.Generator().manual_seed(CFG["seed"]))
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], num_workers=2)
print(f"train {len(train_ds)} / val {len(val_ds)}")

In [ ]:
class TerrainVAE(pl.LightningModule):
    def __init__(self, img_size=64, latent_dim=5, beta=4.0, kl_warmup_epochs=15, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.beta, self.kl_warmup_epochs, self.lr = beta, kl_warmup_epochs, lr

        # Encoder: 64 -> 32 -> 16 -> 8 -> 4 -> 2  (channels 1->32->64->128->128->128)
        enc, c = [], 1
        for out_c in [32, 64, 128, 128, 128]:
            enc += [nn.Conv2d(c, out_c, 4, 2, 1), nn.BatchNorm2d(out_c), nn.ReLU()]
            c = out_c
        self.encoder_conv = nn.Sequential(*enc)

        with torch.no_grad():
            conv_out = self.encoder_conv(torch.zeros(1, 1, img_size, img_size))
        self.conv_shape = conv_out.shape[1:]          # (128, 2, 2)
        flat = int(np.prod(self.conv_shape))         # 512

        # THE BOTTLENECK: conv features -> mu / logvar -> 5-dim latent vector
        self.fc_mu     = nn.Linear(flat, latent_dim)
        self.fc_logvar = nn.Linear(flat, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, flat)

        # Decoder: (128,2,2) -> ... -> (1,64,64), outputs LOGITS (no sigmoid here)
        dec, pairs = [], [(128, 128), (128, 128), (128, 64), (64, 32), (32, 1)]
        for i, (ic, oc) in enumerate(pairs):
            dec.append(nn.ConvTranspose2d(ic, oc, 4, 2, 1))
            if i < len(pairs) - 1:
                dec += [nn.BatchNorm2d(oc), nn.ReLU()]
        self.decoder_conv = nn.Sequential(*dec)

    def encode(self, x):
        h = self.encoder_conv(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)

    def decode(self, z):
        return self.decoder_conv(self.fc_decode(z).view(-1, *self.conv_shape))

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar

    def _beta_now(self):
        if self.kl_warmup_epochs <= 0:
            return self.beta
        return self.beta * min(1.0, self.current_epoch / self.kl_warmup_epochs)

    def _loss(self, logits, x, mu, logvar):
        # BCE summed over pixels, averaged over batch  +  β * KL summed over latent dims, averaged over batch.
        # Both terms are per-sample means, so beta actually means something.
        recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / x.size(0)
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
        beta = self._beta_now()
        return recon + beta * kl, recon, kl, beta

    def training_step(self, batch, batch_idx):
        x, _ = batch
        logits, mu, logvar = self(x)
        loss, recon, kl, beta = self._loss(logits, x, mu, logvar)
        self.log_dict({"train_loss": loss, "train_recon": recon, "train_kl": kl, "beta": beta},
                      prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        logits, mu, logvar = self(x)
        loss, recon, kl, _ = self._loss(logits, x, mu, logvar)
        self.log_dict({"val_loss": loss, "val_recon": recon, "val_kl": kl}, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

model = TerrainVAE(**{k: CFG[k] for k in
    ["img_size", "latent_dim", "beta", "kl_warmup_epochs", "lr"]})
print(model)

In [ ]:
ckpt = ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min",
                       filename="terrain-vae-{epoch:02d}-{val_loss:.4f}")
early = EarlyStopping(monitor="val_loss", patience=10, mode="min")
trainer = pl.Trainer(max_epochs=CFG["max_epochs"], accelerator="auto", devices=1,
                     callbacks=[ckpt, early], default_root_dir="/content/terrain_vae")
trainer.fit(model, train_loader, val_loader)
print("best checkpoint:", ckpt.best_model_path)

In [ ]:
import matplotlib.pyplot as plt
model.eval()
xb, _ = next(iter(val_loader))
xb = xb[:8].to(model.device)
with torch.no_grad():
    recon_logits, _, _ = model(xb)
recon = torch.sigmoid(recon_logits).cpu()

fig, ax = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    ax[0, i].imshow(xb[i, 0].cpu(), cmap="terrain"); ax[0, i].axis("off")
    ax[1, i].imshow(recon[i, 0], cmap="terrain");  ax[1, i].axis("off")
ax[0, 0].set_title("input"); ax[1, 0].set_title("recon")
plt.tight_layout(); plt.show()

In [ ]:
# One row per latent dim, sweeping -3..3. If a row barely changes, that dim collapsed:
# raise beta slightly or train longer. If one slider changes everything, dims are entangled:
# raise beta more (at the cost of blurrier recons).
model.eval()
steps = 9
fig, axes = plt.subplots(CFG["latent_dim"], steps, figsize=(2 * steps, 2 * CFG["latent_dim"]))
with torch.no_grad():
    for d in range(CFG["latent_dim"]):
        for j, v in enumerate(torch.linspace(-3, 3, steps)):
            z = torch.zeros(1, CFG["latent_dim"], device=model.device)
            z[0, d] = v
            img = torch.sigmoid(model.decode(z))[0, 0].cpu()
            axes[d, j].imshow(img, cmap="terrain"); axes[d, j].axis("off")
        axes[d, 0].set_ylabel(f"z[{d}]", rotation=0, labelpad=20, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
import imageio
model.eval()
xa, _ = next(iter(val_loader)); xa = xa.to(model.device)
with torch.no_grad():
    mua, _ = model.encode(xa[:1]); mub, _ = model.encode(xa[1:2])
    frames = []
    for t in np.linspace(0, 1, 30):
        z = (1 - t) * mua + t * mub
        img = torch.sigmoid(model.decode(z))[0, 0].cpu().numpy()
        frames.append((img * 255).astype(np.uint8))
imageio.mimsave("/content/morph.gif", frames, fps=10)
print("saved /content/morph.gif")
plt.imshow(frames[15], cmap="terrain"); plt.axis("off"); plt.title("mid-morph"); plt.show()

In [ ]:
!pip install -q onnx onnxruntime onnxscript

In [ ]:
class DecoderOnly(nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae
    def forward(self, z):                       # z: (1, 5), fixed batch
        h = self.vae.fc_decode(z).reshape(1, *self.vae.conv_shape)  # explicit dims, no -1
        return torch.sigmoid(self.vae.decoder_conv(h))              # (1, 1, 64, 64)

dec = DecoderOnly(model).eval()
dummy = torch.randn(1, CFG["latent_dim"])
torch.onnx.export(dec, dummy, "/content/terrain_decoder.onnx",
                  input_names=["latent"], output_names=["heightmap"],
                  opset_version=14)  # no dynamic_axes this time

import onnx
m = onnx.load("/content/terrain_decoder.onnx")
onnx.save_model(m, "/content/terrain_decoder_single.onnx", save_as_external_data=False)

import onnxruntime as ort
sess = ort.InferenceSession("/content/terrain_decoder_single.onnx", providers=["CPUExecutionProvider"])
z = np.random.randn(1, CFG["latent_dim"]).astype(np.float32)
print("onnx out shape:", sess.run(["heightmap"], {"latent": z})[0].shape)  # expect (1, 1, 64, 64)


In [ ]:
# 16-bit PNGs preserve elevation steps; in Godot read as FORMAT_RF/RH float texture for the shader.
from PIL import Image as PILImage
os.makedirs("/content/godot_heightmaps", exist_ok=True)
model.eval()
with torch.no_grad():
    for i in range(4):
        z = torch.randn(1, CFG["latent_dim"], device=model.device) * 1.5
        h = torch.sigmoid(model.decode(z))[0, 0].cpu().numpy()
        PILImage.fromarray((h * 65535).astype(np.uint16)).save(f"/content/godot_heightmaps/h_{i}.png")
print("saved 4 test heightmaps to /content/godot_heightmaps/")

In [ ]:
!zip -r all_files.zip /content/* -x "*.ipynb"


In [ ]:
import onnx
m = onnx.load("/content/terrain_decoder.onnx")  # picks up the .data automatically
onnx.save_model(m, "/content/terrain_decoder_single.onnx", save_as_external_data=False)

#For More Grovy Mountains

In [ ]:
!python /kaggle/input/datasets/sohampansare/tile-scraper/tile_scraper.py --regions himalayas andes alps --zoom 12 --patch-size 128 --target-patches 4000 --max-tiles 3000 --out /kaggle/working/data128


In [ ]:
!pip install -q lightning onnx onnxruntime imageio  # uncomment if imports fail

import os, math, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
CFG = dict(
    patches_path = "/kaggle/working/data128/patches.npy",  # upload the scraper output here
    img_size     = 128,
    latent_dim   = 5,     # 5 sliders in Godot. If morphs look entangled, try 16 first to confirm the pipeline, then come back to 5.
    beta         = 2.0,   # β-VAE weight: higher = more disentangled sliders, but blurrier reconstructions
    kl_warmup_epochs = 15,
    lr           = 1e-3,
    batch_size   = 64,
    max_epochs   = 80,
    seed         = 42,
    val_frac     = 0.1,
)

def seed_all(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["seed"])

In [ ]:
if not os.path.exists(CFG["patches_path"]):
    from google.colab import files
    print("Upload patches.npy from the tile scraper:")
    files.upload()  # then move it to /content/ if needed

raw = np.load(CFG["patches_path"]).astype(np.float32)  # (N, H, W) in METRES
print("loaded:", raw.shape, f"elev {raw.min():.0f}..{raw.max():.0f} m")

# Per-patch min-max -> [0, 1]. Each patch uses its full dynamic range,
# so every latent position decodes to terrain with real relief (no flat outputs).
mn = raw.reshape(len(raw), -1).min(1)[:, None, None]
mx = raw.reshape(len(raw), -1).max(1)[:, None, None]
rng = np.clip(mx - mn, 1e-6, None)
X = (raw - mn) / rng
X = X[(mx - mn).ravel() > 1.0]  # drop degenerate patches
print("kept patches:", len(X))

X = torch.from_numpy(X[:, None])  # (N, 1, H, W)
ds = TensorDataset(X, X)
n_val = max(1, int(len(ds) * CFG["val_frac"]))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val],
                                generator=torch.Generator().manual_seed(CFG["seed"]))
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], num_workers=2)
print(f"train {len(train_ds)} / val {len(val_ds)}")

In [ ]:
class TerrainVAE(pl.LightningModule):
    def __init__(self, img_size=64, latent_dim=5, beta=4.0, kl_warmup_epochs=15, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.beta, self.kl_warmup_epochs, self.lr = beta, kl_warmup_epochs, lr

        # Encoder: 64 -> 32 -> 16 -> 8 -> 4 -> 2  (channels 1->32->64->128->128->128)
        enc, c = [], 1
        for out_c in [32, 64, 128, 128, 128]:
            enc += [nn.Conv2d(c, out_c, 4, 2, 1), nn.BatchNorm2d(out_c), nn.ReLU()]
            c = out_c
        self.encoder_conv = nn.Sequential(*enc)

        with torch.no_grad():
            conv_out = self.encoder_conv(torch.zeros(1, 1, img_size, img_size))
        self.conv_shape = conv_out.shape[1:]          # (128, 2, 2)
        flat = int(np.prod(self.conv_shape))         # 512

        # THE BOTTLENECK: conv features -> mu / logvar -> 5-dim latent vector
        self.fc_mu     = nn.Linear(flat, latent_dim)
        self.fc_logvar = nn.Linear(flat, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, flat)

        # Decoder: (128,2,2) -> ... -> (1,64,64), outputs LOGITS (no sigmoid here)
        dec, pairs = [], [(128, 128), (128, 128), (128, 64), (64, 32), (32, 1)]
        for i, (ic, oc) in enumerate(pairs):
            dec.append(nn.ConvTranspose2d(ic, oc, 4, 2, 1))
            if i < len(pairs) - 1:
                dec += [nn.BatchNorm2d(oc), nn.ReLU()]
        self.decoder_conv = nn.Sequential(*dec)

    def encode(self, x):
        h = self.encoder_conv(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)

    def decode(self, z):
        return self.decoder_conv(self.fc_decode(z).view(-1, *self.conv_shape))

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar

    def _beta_now(self):
        if self.kl_warmup_epochs <= 0:
            return self.beta
        return self.beta * min(1.0, self.current_epoch / self.kl_warmup_epochs)

    def _loss(self, logits, x, mu, logvar):
        # BCE summed over pixels, averaged over batch  +  β * KL summed over latent dims, averaged over batch.
        # Both terms are per-sample means, so beta actually means something.
        recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / x.size(0)
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
        beta = self._beta_now()
        return recon + beta * kl, recon, kl, beta

    def training_step(self, batch, batch_idx):
        x, _ = batch
        logits, mu, logvar = self(x)
        loss, recon, kl, beta = self._loss(logits, x, mu, logvar)
        self.log_dict({"train_loss": loss, "train_recon": recon, "train_kl": kl, "beta": beta},
                      prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        logits, mu, logvar = self(x)
        loss, recon, kl, _ = self._loss(logits, x, mu, logvar)
        self.log_dict({"val_loss": loss, "val_recon": recon, "val_kl": kl}, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

model = TerrainVAE(**{k: CFG[k] for k in
    ["img_size", "latent_dim", "beta", "kl_warmup_epochs", "lr"]})
print(model)

In [ ]:
ckpt = ModelCheckpoint(monitor="val_recon", save_top_k=1, mode="min",
                       filename="terrain-vae-{epoch:02d}-{val_recon:.4f}")
early = EarlyStopping(monitor="val_recon", patience=15, mode="min")
trainer = pl.Trainer(max_epochs=CFG["max_epochs"], accelerator="auto", devices=1,
                     callbacks=[ckpt, early], default_root_dir="/content/terrain_vae")
trainer.fit(model, train_loader, val_loader)
print("best checkpoint:", ckpt.best_model_path)

In [ ]:
import matplotlib.pyplot as plt
model.eval()
xb, _ = next(iter(val_loader))
xb = xb[:8].to(model.device)
with torch.no_grad():
    recon_logits, _, _ = model(xb)
recon = torch.sigmoid(recon_logits).cpu()

fig, ax = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    ax[0, i].imshow(xb[i, 0].cpu(), cmap="terrain"); ax[0, i].axis("off")
    ax[1, i].imshow(recon[i, 0], cmap="terrain");  ax[1, i].axis("off")
ax[0, 0].set_title("input"); ax[1, 0].set_title("recon")
plt.tight_layout(); plt.show()

In [ ]:
# One row per latent dim, sweeping -3..3. If a row barely changes, that dim collapsed:
# raise beta slightly or train longer. If one slider changes everything, dims are entangled:
# raise beta more (at the cost of blurrier recons).
model.eval()
steps = 9
fig, axes = plt.subplots(CFG["latent_dim"], steps, figsize=(2 * steps, 2 * CFG["latent_dim"]))
with torch.no_grad():
    for d in range(CFG["latent_dim"]):
        for j, v in enumerate(torch.linspace(-3, 3, steps)):
            z = torch.zeros(1, CFG["latent_dim"], device=model.device)
            z[0, d] = v
            img = torch.sigmoid(model.decode(z))[0, 0].cpu()
            axes[d, j].imshow(img, cmap="terrain"); axes[d, j].axis("off")
        axes[d, 0].set_ylabel(f"z[{d}]", rotation=0, labelpad=20, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
import imageio
model.eval()
xa, _ = next(iter(val_loader)); xa = xa.to(model.device)
with torch.no_grad():
    mua, _ = model.encode(xa[:1]); mub, _ = model.encode(xa[1:2])
    frames = []
    for t in np.linspace(0, 1, 30):
        z = (1 - t) * mua + t * mub
        img = torch.sigmoid(model.decode(z))[0, 0].cpu().numpy()
        frames.append((img * 255).astype(np.uint8))
imageio.mimsave("/content/morph.gif", frames, fps=10)
print("saved /content/morph.gif")
plt.imshow(frames[15], cmap="terrain"); plt.axis("off"); plt.title("mid-morph"); plt.show()

In [ ]:
!pip install -q onnx onnxruntime onnxscript

In [ ]:
# Fixed batch=1 (matches Godot usage), explicit reshape dims (DirectML-safe),
# opset 18 (what torch exports natively - no lossy down-conversion).
class DecoderOnly(nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae
    def forward(self, z):                       # z: (1, latent_dim) in [-3, 3]
        h = self.vae.fc_decode(z).reshape(1, *self.vae.conv_shape)
        return torch.sigmoid(self.vae.decoder_conv(h))  # (1, 1, H, W) in [0, 1]

dec = DecoderOnly(model).eval()
dummy = torch.randn(1, CFG["latent_dim"])
onnx_path = "/kaggle/working/terrain_decoder.onnx"
torch.onnx.export(dec, dummy, onnx_path,
                  input_names=["latent"], output_names=["heightmap"],
                  opset_version=18)
print("exported", onnx_path, f"{os.path.getsize(onnx_path)/1024:.0f} KB")

# Merge weights into one file: Godot's importer ignores the .onnx.data sidecar.
import onnx
m = onnx.load(onnx_path)
single_path = "/kaggle/working/terrain_decoder_single.onnx"
onnx.save_model(m, single_path, save_as_external_data=False)
print("single-file:", single_path, f"{os.path.getsize(single_path)/1024:.0f} KB")

# Verify the ONNX graph agrees with PyTorch
import onnxruntime as ort
sess = ort.InferenceSession(single_path, providers=["CPUExecutionProvider"])
z = np.random.randn(1, CFG["latent_dim"]).astype(np.float32)
with torch.no_grad():
    ref = dec(torch.from_numpy(z)).numpy()
got = sess.run(["heightmap"], {"latent": z})[0]
print("onnx out shape:", got.shape)
print("max abs diff pytorch vs onnx:", np.abs(ref - got).max())


In [ ]:
# 16-bit PNGs preserve elevation steps; in Godot read as FORMAT_RF/RH float texture for the shader.
from PIL import Image as PILImage
os.makedirs("/kaggle/working//godot_heightmaps", exist_ok=True)
model.eval()
with torch.no_grad():
    for i in range(4):
        z = torch.randn(1, CFG["latent_dim"], device=model.device) * 1.5
        h = torch.sigmoid(model.decode(z))[0, 0].cpu().numpy()
        PILImage.fromarray((h * 65535).astype(np.uint16)).save(f"/content/godot_heightmaps/h_{i}.png")
print("saved 4 test heightmaps to /content/godot_heightmaps/")

In [ ]:
!zip -r all_files.zip /content/* -x "*.ipynb"